In [ ]:
import numpy as np
import pandas as pd
import sys
import os
sys.path.append(os.path.abspath('../..'))
import matplotlib.pyplot as plt
from scripts import nodes as n
from scripts import elements as e
from scripts import material_params as mat
from scipy.linalg import eigh
import plotly.graph_objects as go

In [ ]:
nodes = []
nodal_values = np.loadtxt('../../text_files/nodes_minimal_model_.txt', delimiter=',')
for i in range(nodal_values.shape[0]):
  nodes.append(n.nodes(nodal_values[i, 0], nodal_values[i, 1], nodal_values[i, 2]))

In [ ]:
# Truss parameters
E = 210e9  # Young's modulus in Pascals
nu = 0.3   # Poisson's ratio
G = E / (2 * (1 + nu))  # Shear modulus in Pascals
m = 5596024 #kg
d1 = 0.8 #m
d2 = 1.8 #m
t1 = 0.03 #m
t2 = 0.08 #m
L_truss_element_y = 15 #m
h_truss = 18 #m
Aeq, Ieqy, Ieqz, b_eq, h_eq = mat.effective_truss_stiffness(d1, d2, t1, t2, h_truss, L_truss_element_y) 
L = 237.5
rho_truss = m / (L * Aeq)
It = (b_eq * h_eq**3 / 3) * (1 - 0.63 * (h_eq / b_eq) * (1 - (h_eq**4 / (12 * b_eq**4)))) *0.06
k = 0.08
Ip  = Ieqy + Ieqz
ep_K = [E, G, Aeq, Ieqy, Ieqz, It, k]
ep_m = [rho_truss, Aeq, Ieqy, Ieqz, Ip]

In [ ]:
k = 5/6
rho = 7850 #kg/m^3
E=210e9 #Pa
G = E/(2*(1+nu)) #Pa
nu = 0.3 #Poisson's ratio
k_fender = mat.stiffness_fenders()
Iy_connect, Iz_connect, Ip_connect, It_connect, A_connect = mat.stiffness_connecting_beams()
ep_K_connect = [E, G, A_connect, Iy_connect, Iz_connect, It_connect, k]
ep_m_connect = [rho, A_connect, Iy_connect, Iz_connect, Ip_connect]


E = 210e9 #Pa
# Material parameters retaining wall
t_eff = 1#m


A_eq_wall, I_eqy_wall, I_eqz_wall, b_eq_wall, rho_wall = mat.effective_retaining_wall_stiffness(t_eff)
Ip_wall = I_eqy_wall + I_eqz_wall
It_wall = (b_eq_wall * t_eff**3 / 3) * (1 - 0.63 * (b_eq_wall / t_eff) + 0.052 * (b_eq_wall / t_eff) **5)

I_eqy_wall = 1e7
I_eqz_wall = 1e7
Ip_wall = I_eqy_wall + I_eqz_wall
It_wall  = 1e7

ep_K_wall = [E, G, A_eq_wall, I_eqy_wall, I_eqz_wall, It_wall, k]
ep_m_wall = [rho_wall, A_eq_wall, I_eqy_wall, I_eqz_wall, Ip_wall]
print(rho_wall)

print(mat.effective_retaining_wall_stiffness(t_eff))
print(mat.effective_truss_stiffness(d1, d2, t1, t2, L_truss_element_y, h_truss))
print(mat.stiffness_connecting_beams())

In [ ]:
elements = []
element_nodes = np.loadtxt('../../text_files/element_nodes3.txt', dtype=int)
for i in range(element_nodes.shape[0]):
    if element_nodes[i, 2] == 0:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K, ep_m))
    elif element_nodes[i, 2] == 1:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_wall, ep_m_wall))
    elif element_nodes[i, 2] == 2:
        elements.append(e.elements(nodes[element_nodes[i, 0] - 1], nodes[element_nodes[i, 1] - 1], ep_K_connect, ep_m_connect))

element_nodes = np.loadtxt('../../text_files/element_nodes.txt', dtype=int)


dofs = n.degrees_of_freedom(nodes)

element_locs = []

for (nA, nB) in element_nodes:
    dofs_A = dofs[f'dof_{nA}']
    dofs_B = dofs[f'dof_{nB}']
    element_locs.append(np.hstack((dofs_A, dofs_B)))



In [ ]:
N = len(nodes)
DOFS_per_node = 6



K_global = np.zeros((N * DOFS_per_node, N * DOFS_per_node))
M_global = np.zeros((N * DOFS_per_node, N * DOFS_per_node))

K_locs = []
M_locs = []



for i in range(len(element_locs)):
    K_global[np.ix_(element_locs[i], element_locs[i])] += elements[i][-1]
    M_global[np.ix_(element_locs[i], element_locs[i])] += elements[i][-2]



# To restore symmetry of the global stiffness and mass matrices
K_global = 0.5 * (K_global + K_global.T)
M_global = 0.5 * (M_global + M_global.T)



In [ ]:
fender_dofs = [70, 75, 79, 83, 87, 91, 95, 99, 103, 107, 111, 115, 119, 123, 128]

k_fender_dofs = [dofs[f'dof_{i+1}'][2] for i in fender_dofs]
for dof in k_fender_dofs:
    K_global[dof, dof] += k_fender



In [ ]:
indices_to_remove = np.hstack((dofs['dof_1'][0:3], dofs['dof_129'][0:2]))
keep_indices = np.setdiff1d(np.arange(N * DOFS_per_node), indices_to_remove)
K_global_reduced = K_global[np.ix_(keep_indices, keep_indices)]
M_global_reduced = M_global[np.ix_(keep_indices, keep_indices)]

In [ ]:
eigvals_global, eigvecs_global = eigh(K_global_reduced, M_global_reduced)


tol = 1e-6
positive = eigvals_global > tol
eigvals_global = eigvals_global[positive]
eigvecs_global = eigvecs_global[:, positive]

frequencies_rad = np.sqrt(eigvals_global)
frequencies_hz = frequencies_rad / (2 * np.pi)


data_freqs = pd.DataFrame({
    'Frequency (Hz)': frequencies_hz,
    'Frequency (rad/s)': frequencies_rad
})

display(data_freqs[:15])


In [ ]:
eigvecs_full = e.expand_eigenvectors(eigvecs_global, keep_indices, N*DOFS_per_node)
print("Expanded eigenvectors shape:", eigvecs_full.shape)

In [ ]:
eigvecs_disp_full = e.extract_displacement(eigvecs_full, keep=3, skip=3)
eigvecs_full.shape


In [ ]:
#First ten modes
modes_to_plot = 4
modes = []
for i in range(modes_to_plot):
    modes.append(eigvecs_disp_full[:, i].reshape(-1, 3) + nodes)


In [ ]:

for i in range(modes_to_plot):
    line_x = []
    line_y = []
    line_z = []
    for nA, nB in element_nodes:
        p1 = modes[i][nA - 1]
        p2 = modes[i][nB - 1]
        line_x.extend([p1[0], p2[0], None])
        line_y.extend([p1[1], p2[1], None])
        line_z.extend([p1[2], p2[2], None])

    fig_lines = go.Figure(
        data=[
            go.Scatter3d(
                x=line_x,
                y=line_y,
                z=line_z,
                mode='lines',
                line=dict(color='black', width=2)
            ),
            go.Scatter3d(
                x=modes[i][:, 0],
                y=modes[i][:, 1],
                z=modes[i][:, 2],
                mode='markers',
                marker=dict(size=3, color='blue')
            )
        ]
    )

    # X = modes[i][:, 0]
    # Y = modes[i][:, 1]
    # Z = modes[i][:, 2]
    # e.set_equal_aspect(fig_lines, X, Y, Z)

    fig_lines.update_layout(
        title=f'Mode {i+1} - Deformed Configuration (Elements)',
        scene=dict(
            xaxis_title='X (m)',
            yaxis_title='Y (m)',
            zaxis_title='Z (m)'
        ),
        width=900,
        height=700
    )

    fig_lines.show()

In [ ]:
# Modal deformations of the truss in 2D
indices_left = list([0]) + list(range(5, 24)) + list(range(43, 52))
indices_right = list([0]) + list(range(24, 43)) + list(range(52, 61))
indices_wall = (list(range(70, 84)) +[1] + list(range(84, 105)) + [2] +list(range(105, 129)))

truss_nodes_left = [nodes[i] for i in range(len(indices_left))]
truss_nodes_right = [nodes[i] for i in range(len(indices_right))]
retaining_wall_nodes = [nodes[i] for i in range(len(indices_wall))]


In [ ]:
modes = []
for i in range(modes_to_plot):
    modes.append(eigvecs_disp_full[:, i].reshape(-1, 3))
modes = np.array(modes)              

modes_truss_left = modes[:, indices_left, :] 
modes_truss_right = modes[:, indices_right, :]
modes_wall = modes[:, indices_wall, :]


# Displacement of truss on the left side

In [ ]:
fig, axes = plt.subplots(modes_to_plot, 3, figsize=(18, 3*modes_to_plot))

if modes_to_plot == 1:
    axes = axes.reshape(1, 3)

x_undeformed = np.array([node[0] for node in truss_nodes_left])
z_undeformed = np.array([node[2] for node in truss_nodes_left])

for i in range(modes_to_plot):
    x_deformed = modes_truss_left[i][:, 0]
    y_deformed = modes_truss_left[i][:, 1]
    z_deformed = modes_truss_left[i][:, 2]

    # shared y-limit for this row (mode)
    row_max = max(
        np.max(np.abs(x_deformed)),
        np.max(np.abs(y_deformed)),
        np.max(np.abs(z_deformed))
    )
    row_max = 1.1 * row_max if row_max > 0 else 1e-12

    # X
    ax_x = axes[i, 0]
    ax_x.plot(np.zeros_like(x_undeformed), 'k--', alpha=0.5)
    ax_x.plot(x_deformed, 'r-', marker='o', markersize=4)
    ax_x.set_title(f'Mode {i+1} - X Displacement ({frequencies_hz[i]:.2f} Hz)')
    ax_x.set_ylim(-row_max, row_max)
    ax_x.grid(True, alpha=0.3)

    # Y
    ax_y = axes[i, 1]
    ax_y.plot(np.zeros_like(x_undeformed), 'k--', alpha=0.5)
    ax_y.plot(y_deformed, 'g-', marker='o', markersize=4)
    ax_y.set_title(f'Mode {i+1} - Y Displacement ({frequencies_hz[i]:.2f} Hz)')
    ax_y.set_ylim(-row_max, row_max)
    ax_y.grid(True, alpha=0.3)

    # Z
    ax_z = axes[i, 2]
    ax_z.plot(z_undeformed, 'k--', alpha=0.5)
    ax_z.plot(z_deformed, 'b-', marker='o', markersize=4)
    ax_z.set_title(f'Mode {i+1} - Z Displacement ({frequencies_hz[i]:.2f} Hz)')
    ax_z.set_ylim(-row_max, row_max)
    ax_z.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Displacement of the truss on the right side

In [ ]:
fig, axes = plt.subplots(modes_to_plot, 3, figsize=(18, 3*modes_to_plot))

if modes_to_plot == 1:
    axes = axes.reshape(1, 3)

x_undeformed = np.array([node[0] for node in truss_nodes_right])
z_undeformed = np.array([node[2] for node in truss_nodes_right])

for i in range(modes_to_plot):
    x_deformed = modes_truss_right[i][:, 0]
    y_deformed = modes_truss_right[i][:, 1]
    z_deformed = modes_truss_right[i][:, 2]

    # shared y-limit for this row (mode)
    row_max = max(
        np.max(np.abs(x_deformed)),
        np.max(np.abs(y_deformed)),
        np.max(np.abs(z_deformed))
    )
    row_max = 1.1 * row_max if row_max > 0 else 1e-12

    # X
    ax_x = axes[i, 0]
    ax_x.plot(np.zeros_like(x_undeformed), 'k--', alpha=0.5)
    ax_x.plot(x_deformed, 'r-', marker='o', markersize=4)
    ax_x.set_title(f'Mode {i+1} - X Displacement ({frequencies_hz[i]:.2f} Hz)')
    ax_x.set_ylim(-row_max, row_max)
    ax_x.grid(True, alpha=0.3)

    # Y
    ax_y = axes[i, 1]
    ax_y.plot(np.zeros_like(x_undeformed), 'k--', alpha=0.5)
    ax_y.plot(y_deformed, 'g-', marker='o', markersize=4)
    ax_y.set_title(f'Mode {i+1} - Y Displacement ({frequencies_hz[i]:.2f} Hz)')
    ax_y.set_ylim(-row_max, row_max)
    ax_y.grid(True, alpha=0.3)

    # Z
    ax_z = axes[i, 2]
    ax_z.plot(z_undeformed, 'k--', alpha=0.5)
    ax_z.plot(z_deformed, 'b-', marker='o', markersize=4)
    ax_z.set_title(f'Mode {i+1} - Z Displacement ({frequencies_hz[i]:.2f} Hz)')
    ax_z.set_ylim(-row_max, row_max)
    ax_z.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Modes retaining wall

In [ ]:
fig, axes = plt.subplots(modes_to_plot, 3, figsize=(18, 3 * modes_to_plot))

if modes_to_plot == 1:
    axes = axes.reshape(1, 3)

retaining_wall_nodes_arr = np.asarray(retaining_wall_nodes)
x_undeformed = retaining_wall_nodes_arr[:, 0]
z_undeformed = retaining_wall_nodes_arr[:, 2]

for i in range(modes_to_plot):
    x_deformed = modes_wall[i][:, 0]
    y_deformed = modes_wall[i][:, 1]
    z_deformed = modes_wall[i][:, 2]

    row_max = max(
        np.max(np.abs(x_deformed)),
        np.max(np.abs(y_deformed)),
        np.max(np.abs(z_deformed))
    )
    row_max = 1.1 * row_max if row_max > 0 else 1e-12

    # X
    ax_x = axes[i, 0]
    ax_x.plot(np.zeros_like(x_undeformed), "k--", alpha=0.5)
    ax_x.plot(x_deformed, "r-", marker="o", markersize=4)
    ax_x.set_title(f"Mode {i+1} - X Displacement ({frequencies_hz[i]:.2f} Hz)")
    ax_x.set_ylim(-row_max, row_max)
    ax_x.grid(True, alpha=0.3)

    # Y
    ax_y = axes[i, 1]
    ax_y.plot(np.zeros_like(x_undeformed), "k--", alpha=0.5)
    ax_y.plot(y_deformed, "g-", marker="o", markersize=4)
    ax_y.set_title(f"Mode {i+1} - Y Displacement ({frequencies_hz[i]:.2f} Hz)")
    ax_y.set_ylim(-row_max, row_max)
    ax_y.grid(True, alpha=0.3)

    # Z
    ax_z = axes[i, 2]
    ax_z.plot(z_undeformed, "k--", alpha=0.5)
    ax_z.plot(z_deformed, "b-", marker="o", markersize=4)
    ax_z.set_title(f"Mode {i+1} - Z Displacement ({frequencies_hz[i]:.2f} Hz)")
    ax_z.set_ylim(-row_max, row_max)
    ax_z.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
#First ten modes
modes_to_plot = 10


In [ ]:
# Animation settings
num_frames = 180
t = np.linspace(0, 6*np.pi, num_frames)
omega = 1.0   # rad/s

# modes_to_plot is already defined
# eigvecs_disp_full is your modal displacement matrix
# nodes is your fixed nodal coordinates (n_nodes, 3)

nodes = np.array(nodes)  
eigvecs_disp_full = np.array(eigvecs_disp_full)

for i in range(modes_to_plot):

    # Mode shape displacements only (do NOT include nodes here)
    mode_disp = eigvecs_disp_full[:, i].reshape(-1, 3)

    # ---- FIXED AXES RANGES ----
    X = nodes[:,0] + mode_disp[:,0]
    Y = nodes[:,1] + mode_disp[:,1]
    Z = nodes[:,2] + mode_disp[:,2]

    # symmetric ranges
    max_range = max(
        np.max(np.abs(X - nodes[:,0])),
        np.max(np.abs(Y - nodes[:,1])),
        np.max(np.abs(Z - nodes[:,2]))
    )
    x_range = [nodes[:,0].min() - max_range, nodes[:,0].max() + max_range]
    y_range = [nodes[:,1].min() - max_range, nodes[:,1].max() + max_range]
    z_range = [nodes[:,2].min() - max_range, nodes[:,2].max() + max_range]

    frames = []
    for k in range(num_frames):

        # Animated displacement (sinusoidal)
        U = nodes + np.sin(omega * t[k]) * mode_disp  # nodes stay constant

        # Build element lines
        line_x, line_y, line_z = [], [], []
        for nA, nB in element_nodes:
            p1 = U[nA - 1]
            p2 = U[nB - 1]
            line_x += [p1[0], p2[0], None]
            line_y += [p1[1], p2[1], None]
            line_z += [p1[2], p2[2], None]

        frames.append(
            go.Frame(
                data=[
                    go.Scatter3d(
                        x=line_x, y=line_y, z=line_z,
                        mode="lines",
                        line=dict(color="black", width=3)
                    ),
                    go.Scatter3d(
                        x=U[:,0], y=U[:,1], z=U[:,2],
                        mode="markers",
                        marker=dict(size=3, color="blue")
                    )
                ]
            )
        )

    # INITIAL FIGURE
    fig = go.Figure(
        data=frames[0].data,
        frames=frames
    )

    fig.update_layout(
        title=f"Mode {i+1} – Animated Mode Shape (Nodes Fixed)",
        width=900,
        height=700,
        scene=dict(
            xaxis=dict(title="X", range=x_range),
            yaxis=dict(title="Y", range=y_range),
            zaxis=dict(title="Z", range=z_range)
        ),
        updatemenus=[
            {
                "type": "buttons",
                "buttons": [
                    {
                        "label": "Play",
                        "method": "animate",
                        "args": [
                            None,
                            {"frame": {"duration": 50, "redraw": True},
                             "transition": {"duration": 0}}
                        ]
                    },
                    {
                        "label": "Pause",
                        "method": "animate",
                        "args": [
                            [None],
                            {"mode": "immediate"}
                        ]
                    }
                ]
            }
        ]
    )

    fig.show()